# MichiganCast Demo 03 — Multimodal Training and Tuning（训练 + 调参）

本 notebook 用于展示：

1. 多模态模型训练工程化入口（CLI + 可复现实验）。
2. 手动调参矩阵（窗口/hidden/dropout/lr/loss/threshold）。
3. 本地实验追踪与每轮指标可视化。

## 0. 本章目标与结论

**目标：**
1. 读取并展示 `configs/experiments` 手动调参矩阵。
2. 用 `src.models.multimodal.train` 启动训练。
3. 读取 `artifacts/experiments` 和 epoch 曲线产物，展示调参闭环。

**结论模板（执行后补充）：**
- 哪组配置在验证集指标上更优。
- 调参对 PR-AUC/F1/Recall 的主要影响方向。
- 训练效率与稳定性的取舍。

## 1. 输入与输出（路径与产物）

| 类型 | 路径 |
|---|---|
| 调参矩阵配置 | `configs/experiments/t21_exp*.yaml` |
| 训练入口 | `src/models/multimodal/train.py` |
| 每轮指标 CSV | `artifacts/reports/multimodal_epoch_metrics.csv` |
| 每轮曲线图 | `artifacts/reports/multimodal_train_validation_metrics.png` |
| 训练摘要 | `artifacts/reports/multimodal_train_summary.json` |
| 实验 run 目录 | `artifacts/experiments/<run_id>/metrics.json` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import yaml

EXP_DIR = Path('configs/experiments')
REPORT_DIR = Path('artifacts/reports')
RUN_DIR = Path('artifacts/experiments')

print('experiment config dir exists:', EXP_DIR.exists())
print('report dir exists:', REPORT_DIR.exists())
print('experiment run dir exists:', RUN_DIR.exists())

## 2. 方法与实现（调用 `src` 模块）

默认只展示命令；如果要执行训练，把 `DEMO_EXECUTE=True`。

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 读取手动调参矩阵（T21）

读取 `t21_exp*.yaml` 并整理成可对比表。

In [ ]:
exp_files = sorted(EXP_DIR.glob('t21_exp*.yaml'))
print('experiment config count:', len(exp_files))

rows = []
for p in exp_files:
    cfg = yaml.safe_load(p.read_text(encoding='utf-8'))
    rows.append({
        'experiment_id': cfg.get('metadata', {}).get('experiment_id', p.stem),
        'horizon_hours': cfg.get('task', {}).get('horizon_hours'),
        'meteo_lookback': cfg.get('task', {}).get('meteo_lookback_steps'),
        'image_lookback': cfg.get('task', {}).get('image_lookback_steps'),
        'conv_hidden_dim': cfg.get('model', {}).get('conv_hidden_dim'),
        'meteo_hidden_dim': cfg.get('model', {}).get('meteo_hidden_dim'),
        'dropout': cfg.get('model', {}).get('dropout'),
        'learning_rate': cfg.get('train', {}).get('learning_rate'),
        'loss_type': cfg.get('train', {}).get('loss', {}).get('type'),
        'decision_threshold': cfg.get('train', {}).get('decision_threshold'),
    })

df_matrix = pd.DataFrame(rows)
display(df_matrix)

### 2.2 启动多模态训练（T20/T22）

**目的：** 通过 CLI 训练入口运行一次可追踪实验，自动产出 run 目录和 metrics。

In [ ]:
cmd_train = (
    'scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train '
    '--input-csv data/interim/traverse_city_daytime_clean_v1.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output-dir artifacts/reports '
    '--checkpoint-path models/pytorch/michigancast_multimodal_best.pth '
    '--device auto '
    '--apple-metal-opt '
    '--experiment-name demo_tuning '
    '--experiment-root artifacts/experiments'
)
run_or_print(cmd_train)

### 2.3 实验记录系统（T22）

列出最近 run，并读取 `metrics.json`。

In [ ]:
run_dirs = sorted([p for p in RUN_DIR.glob('*') if p.is_dir()])
print('run count:', len(run_dirs))
for p in run_dirs[-5:]:
    print('-', p.name)

if run_dirs:
    latest = run_dirs[-1]
    metrics_path = latest / 'metrics.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        print('\nlatest run id:', metrics.get('run_id'))
        print('best_val_loss:', metrics.get('best_val_loss'))
        print('test_pr_auc:', metrics.get('test_metrics', {}).get('pr_auc'))
    else:
        print('latest run has no metrics.json')

## 3. 结果解读（图表与指标）

读取每轮指标与训练摘要，用于演示模型学习过程。

In [ ]:
epoch_csv = REPORT_DIR / 'multimodal_epoch_metrics.csv'
summary_json = REPORT_DIR / 'multimodal_train_summary.json'
curve_png = REPORT_DIR / 'multimodal_train_validation_metrics.png'

print('epoch csv exists:', epoch_csv.exists())
print('summary exists:', summary_json.exists())
print('curve png exists:', curve_png.exists())

if epoch_csv.exists():
    df_epoch = pd.read_csv(epoch_csv)
    display(df_epoch.tail(10))
else:
    print('No epoch metrics CSV yet. Run section 2.2 first.')

In [ ]:
if summary_json.exists():
    summary = json.loads(summary_json.read_text(encoding='utf-8'))
    print('device:', summary.get('runtime_profile', {}).get('device'))
    print('batch_size:', summary.get('runtime_profile', {}).get('batch_size'))
    print('epochs_ran:', summary.get('fit_result', {}).get('epochs_ran'))
    print('best_val_loss:', summary.get('fit_result', {}).get('best_val_loss'))
    print('test_pr_auc:', summary.get('test_metrics', {}).get('pr_auc'))
else:
    print('No training summary yet. Run section 2.2 first.')

### 3.1 演示讲解要点（建议）

1. 手动调参矩阵如何覆盖“容量、正则化、优化、阈值”四类因素。
2. 训练曲线如何判断过拟合、欠拟合与学习率不匹配。
3. 为什么需要 run 级别追踪（便于复现实验、对比结果、写 portfolio）。

## 4. 工程反思与下一步

- 反思：当前调参仍偏手工，是否需要后续自动化搜索策略。
- 风险：样本不平衡会导致看起来 loss 下降但正类指标不升。
- 下一步：进入 `04_imbalance_stability_export_serve.ipynb` 展示不平衡策略、稳定性、导出与服务化。

## Appendix. 复现实验命令

```bash
scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train --device auto --apple-metal-opt --experiment-name demo_tuning
scripts/run_in_pytorch_env.sh python -m src.models.multimodal.train --device auto --apple-metal-opt --experiment-name demo_tuning_alt --threshold 0.45 --learning-rate 0.0008
```